# Day 05: Structured outputs

**The job:** an accounts team gets invoices as messy text (pasted from PDFs, emails, WhatsApp). Our code must turn each one into clean data, so it can add up totals, check the GST and save it to a database.

The model can read invoices fine. The hard part is getting the answer in **a shape our code can trust**. We ask for JSON three ways:

1. **Prompt only:** "return JSON". The model *may* listen.
2. **JSON mode:** the API guarantees valid JSON, but any shape.
3. **Strict structured outputs:** the API guarantees valid JSON **that matches our schema**.

Same model, same prompt, same 2 invoices. Only the API setting changes.

**Model:** `gpt-4o-mini` via the Responses API, `openai` 3.30, `pydantic` 2.14. Checked 2026-10-14 at https://developers.openai.com/api/docs/guides/structured-outputs. Re-check before class.

LLM output changes from run to run, so yours may differ a little from the saved output.

In [1]:
import json
from typing import Literal

from dotenv import load_dotenv
from openai import BadRequestError, OpenAI
from pydantic import BaseModel, Field, ValidationError

load_dotenv(".env")
client = OpenAI()
MODEL = "gpt-4o-mini"

PROMPT = "Extract this invoice as JSON: vendor, invoice number, date, line items, tax and total."


def ask(document, **kwargs):
    return client.responses.create(model=MODEL, input=f"{PROMPT}\n\n{document}", temperature=0, **kwargs)

## The 2 invoices

One tidy tax invoice from a shop, one casual email from a freelancer. Real inputs look like both.

In [2]:
SHOP_INVOICE = """TAX INVOICE
Sharma Electronics, Nehru Place, New Delhi
Inv No: SE/2026/0417        Date: 03-10-2026
Bill to: Pixel Studio, Pune

1. Dell 24" monitor          x2   @ 13,499
2. Logitech keyboard         x1   @ 9,995
3. HDMI cable (2m)           x3   @ 349
                          Subtotal: 38,040
                          GST 18%:   6,847.20
                       Grand Total: Rs 44,887.20
Payment due in 15 days. UPI / NEFT."""

FREELANCER_EMAIL = """hey! invoice for last month's work:
logo design 15k, 2 social media posts @ 2500 each, website banner 4000.
total 24000, no GST (not registered yet).
pay to Rohan Mehta, inv #RM-12, dated 1st oct 2026. thanks!"""

INVOICES = {"shop": SHOP_INVOICE, "freelancer": FREELANCER_EMAIL}

## The schema: what our code needs

Written as a **Pydantic** model (TypeScript: a **Zod** schema). Note the nesting: an invoice has a **list** of line items.

- `Literal[...]` = only these exact values are allowed.
- `X | None` = the key must be there, but the value can be `null`.
- `Field(description=...)` goes into the schema too. The model reads it, so it works like a mini prompt.

And a helper that does what our real code would do: parse the reply, check it against the schema, then **use** it (add up the line items).

In [3]:
class LineItem(BaseModel):
    description: str
    quantity: int
    unit_price: float


class Invoice(BaseModel):
    vendor: str
    invoice_number: str
    invoice_date: str = Field(description="YYYY-MM-DD")
    currency: Literal["INR", "USD", "EUR"]
    line_items: list[LineItem]
    tax: float | None = Field(description="tax amount, null if no tax")
    total: float


def try_to_use(reply: str):
    """What our program would do with the reply."""
    print("REPLY:", reply[:300] + ("..." if len(reply) > 300 else ""))
    try:
        data = json.loads(reply)
    except json.JSONDecodeError as e:
        print(f"\n✗ json.loads failed: {e}")
        return
    try:
        invoice = Invoice.model_validate(data, strict=True)
    except ValidationError as e:
        print(f"\n✗ valid JSON, but wrong shape ({e.error_count()} problems), e.g.:")
        for err in e.errors()[:4]:
            print("   ", ".".join(map(str, err["loc"])), "→", err["msg"])
        return
    items_sum = sum(i.quantity * i.unit_price for i in invoice.line_items)
    print(f"\n✓ usable! {len(invoice.line_items)} line items add up to {items_sum:,.2f}, "
          f"+ tax {invoice.tax or 0:,.2f} = {items_sum + (invoice.tax or 0):,.2f}  (invoice says {invoice.total:,.2f})")

## Level 1: prompt only

We just *ask* for JSON. Nothing enforces it.

In [4]:
for name, doc in INVOICES.items():
    print(f"=== {name} ===")
    try_to_use(ask(doc).output_text)
    print()

=== shop ===


REPLY: Here’s the extracted invoice in JSON format:

```json
{
  "vendor": "Sharma Electronics, Nehru Place, New Delhi",
  "invoice_number": "SE/2026/0417",
  "date": "03-10-2026",
  "line_items": [
    {
      "description": "Dell 24\" monitor",
      "quantity": 2,
      "unit_price": 13499,
      "total...

✗ json.loads failed: Expecting value: line 1 column 1 (char 0)

=== freelancer ===


REPLY: Here’s the extracted invoice in JSON format:

```json
{
  "vendor": "Rohan Mehta",
  "invoice_number": "RM-12",
  "date": "2026-10-01",
  "line_items": [
    {
      "description": "Logo Design",
      "amount": 15000
    },
    {
      "description": "Social Media Post",
      "amount": 2500,
     ...

✗ json.loads failed: Expecting value: line 1 column 1 (char 0)



The model understood both invoices, but wrapped the JSON in "Here is..." and ```` ```json ```` fences. A program calling `json.loads` crashes on the first character.

## Level 2: JSON mode

One setting: `text={"format": {"type": "json_object"}}`. Now the reply always parses. But *what's inside* is up to the model.

(Quirk: OpenAI returns an error if the word "JSON" isn't in the input. Our prompt has it.)

In [5]:
json_mode = {}
for name, doc in INVOICES.items():
    print(f"=== {name} ===")
    json_mode[name] = ask(doc, text={"format": {"type": "json_object"}}).output_text
    try_to_use(json_mode[name])
    print()

=== shop ===


REPLY: {
  "vendor": {
    "name": "Sharma Electronics",
    "address": "Nehru Place, New Delhi"
  },
  "invoice_number": "SE/2026/0417",
  "date": "03-10-2026",
  "bill_to": "Pixel Studio, Pune",
  "line_items": [
    {
      "description": "Dell 24\" monitor",
      "quantity": 2,
      "unit_price": 134...

✗ valid JSON, but wrong shape (4 problems), e.g.:
    vendor → Input should be a valid string
    invoice_date → Field required
    currency → Field required
    tax → Input should be a valid number

=== freelancer ===


REPLY: {
  "vendor": "Rohan Mehta",
  "invoice_number": "RM-12",
  "date": "2026-10-01",
  "line_items": [
    {
      "description": "Logo Design",
      "amount": 15000
    },
    {
      "description": "Social Media Post",
      "amount": 2500,
      "quantity": 2
    },
    {
      "description": "Webs...

✗ valid JSON, but wrong shape (8 problems), e.g.:
    invoice_date → Field required
    currency → Field required
    line_items.0.quantity → Field required
    line_items.0.unit_price → Field required



Now it parses, but the model chose its own shape: `date` instead of `invoice_date` (and not as YYYY-MM-DD), no `currency`, `vendor` as a nested object, `amount` instead of `quantity` + `unit_price`. Worse, **the two invoices came back in different shapes**. Compare the keys:

In [6]:
for name, reply in json_mode.items():
    data = json.loads(reply)
    print(f"{name:<11} top-level keys: {list(data)}")

shop        top-level keys: ['vendor', 'invoice_number', 'date', 'bill_to', 'line_items', 'subtotal', 'tax', 'total']
freelancer  top-level keys: ['vendor', 'invoice_number', 'date', 'line_items', 'tax', 'total']


You can't write code against a shape that changes per invoice.

## Level 3: strict structured outputs

Pass the Pydantic class itself: `client.responses.parse(..., text_format=Invoice)`. The SDK turns it into a JSON Schema, and the API makes the model's output match it. You get `response.output_parsed`: a real `Invoice` object.

In [7]:
for name, doc in INVOICES.items():
    print(f"=== {name} ===")
    r = client.responses.parse(model=MODEL, input=f"{PROMPT}\n\n{doc}", temperature=0, text_format=Invoice)
    try_to_use(r.output_text)
    print()

invoice = r.output_parsed
print("A typed object:", invoice.vendor, "|", invoice.invoice_date, "|", invoice.line_items[0])

=== shop ===


REPLY: {
  "vendor": "Sharma Electronics",
  "invoice_number": "SE/2026/0417",
  "invoice_date": "2026-10-03",
  "currency": "INR",
  "line_items": [
    {
      "description": "Dell 24\" monitor",
      "quantity": 2,
      "unit_price": 13499
    },
    {
      "description": "Logitech keyboard",
      "...

✓ usable! 3 line items add up to 38,040.00, + tax 6,847.20 = 44,887.20  (invoice says 44,887.20)

=== freelancer ===


REPLY: {
  "vendor": "Rohan Mehta",
  "invoice_number": "RM-12",
  "invoice_date": "2026-10-01",
  "currency": "INR",
  "line_items": [
    {
      "description": "Logo Design",
      "quantity": 1,
      "unit_price": 15000
    },
    {
      "description": "Social Media Post",
      "quantity": 2,
      ...

✓ usable! 3 line items add up to 24,000.00, + tax 0.00 = 24,000.00  (invoice says 24,000.00)

A typed object: Rohan Mehta | 2026-10-01 | description='Logo Design' quantity=1 unit_price=15000.0


Both invoices, same shape, and our code can do maths on them straight away. The schema did half the prompt's job: the date came back as `YYYY-MM-DD` because the field description said so, and "15k" became `15000.0` because `unit_price` is a number.

**But check the result anyway.** A schema guarantees the *shape*, not the *truth*. "Do the line items add up to the total?" is a check only your code can do. That's why `try_to_use` compares them.

## Shape is not truth: give the model a way out

What if a field simply isn't there? This receipt has no invoice number, but our schema says `invoice_number: str` is **required**.

In [8]:
RECEIPT = "Paid ₹1,200 to Ravi Plumbing for fixing the kitchen tap, 5 Oct 2026. Cash."

r = client.responses.parse(model=MODEL, input=f"{PROMPT}\n\n{RECEIPT}", temperature=0, text_format=Invoice)
print("invoice_number:", repr(r.output_parsed.invoice_number))

invoice_number: 'INV-001'


Forced to fill a required string, the model **made one up** (on our run: `'INV-001'`) instead of saying "there isn't one". A fake invoice number in your accounts is worse than a missing one.

**The fix is in the schema:** make it `str | None`, so `null` is a valid answer.

In [9]:
class SafeInvoice(Invoice):
    invoice_number: str | None = Field(description="null if the document has no invoice number")

r = client.responses.parse(model=MODEL, input=f"{PROMPT}\n\n{RECEIPT}", temperature=0, text_format=SafeInvoice)
print("invoice_number:", repr(r.output_parsed.invoice_number))

invoice_number: None


## Two things strict mode can't save you from

**Cut off.** If the reply hits `max_output_tokens`, the JSON stops halfway. `parse()` then **raises an error** instead of giving you half an invoice. Always handle it.

**Refusal.** If the model refuses (an unsafe request), you get a `refusal` instead of your schema. Check for it before using the result.

In [10]:
try:
    client.responses.parse(model=MODEL, input=f"{PROMPT}\n\n{SHOP_INVOICE}", temperature=0,
                           text_format=Invoice, max_output_tokens=30)
except ValidationError as e:
    print("parse() raised:", e.errors()[0]["msg"])

parse() raised: Invalid JSON: EOF while parsing a string at line 4 column 19


## Takeaways

| | Valid JSON | Right shape | Correct values |
|---|---|---|---|
| Prompt only | maybe | maybe | maybe |
| JSON mode | yes | maybe | maybe |
| **Strict structured outputs** | **yes** | **yes** | maybe |

- If code reads the reply, use **strict structured outputs**. JSON mode is the older, weaker option.
- The schema is part of the prompt: field names, enums and descriptions guide the model.
- Give the model a way out (`| None`), or it will invent an answer.
- Still validate in code (do the totals add up?) and still run evals. A schema guarantees the shape, not the truth.

**Next: tool calling.** Same idea. The model fills in a schema, but this time it's the arguments for *your function*.